In [0]:
import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

logging.basicConfig(level=logging.INFO)

try:
    logging.info("Incremental started")

    # getting last watermark
    last_value = spark.sql("""select coalesce(max(last_unix_time), 0)
        from fraud_catalog.audit.watermark_table""").collect()[0][0]

    logging.info(f"Last watermark: {last_value}")

    # reading bronze and filter new data
    df = spark.table("fraud_catalog.bronze.transactions_raw")

    new_df = df.filter(F.col("unix_time") > last_value)

    new_count = new_df.count()
    logging.info(f"New records: {new_count}")

    # if no new data, stop here
    if new_count == 0:
        logging.info("No new records found")
    else:
        #cleaning new data
        for c in new_df.columns:
            if "unnamed" in c.lower():
                new_df = new_df.drop(c)

        new_df = new_df.withColumnRenamed("first", "first_name") \
                       .withColumnRenamed("last", "last_name") \
                       .withColumnRenamed("long", "cust_long") \
                       .withColumnRenamed("lat", "cust_lat")

        new_df = new_df.withColumn("amt", F.col("amt").cast("double")) \
                       .withColumn("unix_time", F.col("unix_time").cast("long")) \
                       .withColumn("txn_ts", F.from_unixtime("unix_time").cast("timestamp"))

        new_df = new_df.withColumn("txn_date", F.to_date("txn_ts")) \
                       .withColumn("txn_hour", F.hour("txn_ts")) \
                       .withColumn("txn_day", F.date_format("txn_ts", "E")) \
                       .withColumn("is_high_value", F.when(F.col("amt") > 1000, 1).otherwise(0)) \
                       .withColumn("is_night_txn", F.when((F.col("txn_hour") >= 22) | (F.col("txn_hour") <= 5), 1).otherwise(0))

        # removing duplicates before merge
        new_df = new_df.dropDuplicates(["trans_num"])

        logging.info("New data cleaned")

        # merge into silver
        silver_table = DeltaTable.forName(spark, "fraud_catalog.silver.transactions_clean")

        (silver_table.alias("t")
            .merge(new_df.alias("s"),"t.trans_num = s.trans_num")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

        logging.info("Merged into Silver")

        # applying gold logic
        window_day = Window.partitionBy("cc_num", "txn_date")

        new_df = new_df.withColumn("txn_count_per_day", F.count("*").over(window_day)) \
                       .withColumn("rule_multi_txn_day",
                                   F.when(F.col("txn_count_per_day") > 5, 1).otherwise(0))

        new_df = new_df.withColumn("distance_km",6371 * 2 * F.asin(F.sqrt(
                    F.pow(F.sin((F.radians(F.col("merch_lat")) - F.radians(F.col("cust_lat"))) / 2), 2) +
                    F.cos(F.radians(F.col("cust_lat"))) *
                    F.cos(F.radians(F.col("merch_lat"))) *
                    F.pow(F.sin((F.radians(F.col("merch_long")) - F.radians(F.col("cust_long"))) / 2), 2))))

        new_df = new_df.withColumn("rule_distance_anomaly",F.when(F.col("distance_km") > 300, 1).otherwise(0))

        window_time = Window.partitionBy("cc_num").orderBy("txn_ts")

        new_df = new_df.withColumn("prev_txn_ts", F.lag("txn_ts").over(window_time)) \
                       .withColumn("time_diff_sec", F.col("txn_ts").cast("long") - F.col("prev_txn_ts").cast("long")) \
                       .withColumn("rule_rapid_txn",F.when( F.col("prev_txn_ts").isNotNull() & (F.col("time_diff_sec") <= 300),1).otherwise(0))

        new_df = new_df.withColumn("fraud_score",
            F.col("is_high_value") * 30 +
            F.col("is_night_txn") * 10 +
            F.col("rule_multi_txn_day") * 20 +
            F.col("rule_distance_anomaly") * 25 +
            F.col("rule_rapid_txn") * 15)

        new_df = new_df.withColumn("fraud_flag",F.when(F.col("fraud_score") >= 40, 1).otherwise(0))

        new_df = new_df.withColumn("risk_level",
            F.when(F.col("fraud_score") >= 70, "HIGH")
             .when(F.col("fraud_score") >= 40, "MEDIUM")
             .otherwise("LOW"))

        new_df = new_df.withColumn("fraud_reason",F.concat_ws(", ",
                F.when(F.col("is_high_value") == 1, "High Amount"),
                F.when(F.col("is_night_txn") == 1, "Night Transaction"),
                F.when(F.col("rule_multi_txn_day") == 1, "Multiple Txn"),
                F.when(F.col("rule_distance_anomaly") == 1, "Location Mismatch"),
                F.when(F.col("rule_rapid_txn") == 1, "Rapid Txn")))

        new_df = new_df.withColumn("fraud_reason",F.when(F.col("fraud_reason") == "", "Normal").otherwise(F.col("fraud_reason")))

        logging.info("Gold logic applied")

        # make schema same as gold table
        gold_df = spark.table("fraud_catalog.gold.fraud_transactions")

        for col_name, dtype in gold_df.dtypes:
            if col_name not in new_df.columns:
                new_df = new_df.withColumn(col_name, F.lit(None).cast(dtype))

        # reorder columns same as gold table
        new_df = new_df.select(gold_df.columns)

        # merge into gold
        gold_table = DeltaTable.forName(spark, "fraud_catalog.gold.fraud_transactions")

        (gold_table.alias("t").merge(new_df.alias("s"),"t.trans_num = s.trans_num")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

        logging.info("Merged into Gold")

        # update watermark after successful merge
        new_watermark = new_df.agg(F.max("unix_time")).collect()[0][0]

        spark.sql(f"""
        INSERT INTO fraud_catalog.audit.watermark_table
        VALUES ('fraud_pipeline', {new_watermark}, current_timestamp())
        """)

        logging.info(f"Watermark updated: {new_watermark}")
        logging.info("Incremental completed")

except Exception as e:
    logging.error(str(e))
    raise

INFO:root:Incremental started
INFO:root:Last watermark: 1362931730
INFO:root:New records: 0
INFO:root:No new records found
